In [359]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
import pyreadr
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import KBinsDiscretizer





### Carga de datos 

In [360]:

archivo = '../Data/listings.RData'
data_raw = pyreadr.read_r(archivo)
data_airbnb = data_raw['listings'].copy()


In [361]:


print("Dataset shape")
print(data_airbnb.shape)
print()
print("Dataset vista")
display(data_airbnb.head() )

Dataset shape
(171748, 80)

Dataset vista


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,city
0,5456.0,https://www.airbnb.com/rooms/5456,2.025092e+13,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr",Great central location for walking to Convent...,My neighborhood is ideally located if you want...,https://a0.muscache.com/pictures/14084884/b5a3...,8028,...,4.73,4.79,NaN,f,1,1,0,0,3.52,"Austin, Texas"
1,6448.0,https://www.airbnb.com/rooms/6448,2.025092e+13,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & ...","Clean, private space with everything you need ...",The neighborhood is fun and funky (but quiet)!...,https://a0.muscache.com/pictures/airflow/Hosti...,14156,...,4.97,4.88,NaN,t,1,1,0,0,1.98,"Austin, Texas"
2,8502.0,https://www.airbnb.com/rooms/8502,2.025092e+13,2025-09-17,city scrape,Woodland Studio Lodging,Studio rental on lower level of home located i...,,https://a0.muscache.com/pictures/miso/Hosting-...,25298,...,4.69,4.63,NaN,f,1,1,0,0,0.28,"Austin, Texas"
3,13035.0,https://www.airbnb.com/rooms/13035,2.025092e+13,2025-09-17,city scrape,Historic house in highly walkable East Austin,Comfortable 2 bedroom/2 bathroom home very cen...,East Cesar Chavez is a gentrifying urban area ...,https://a0.muscache.com/pictures/miso/Hosting-...,50793,...,5.00,4.95,NaN,f,2,2,0,0,0.11,"Austin, Texas"
4,22828.0,https://www.airbnb.com/rooms/22828,2.025092e+13,2025-09-16,city scrape,Garage Apartment central SE Austin,"Fully furnished, centrally located, second sto...","wikipedia: East_Riverside-Oltorf,_Austin,_Texas",https://a0.muscache.com/pictures/miso/Hosting-...,56488,...,4.72,4.84,NaN,f,1,1,0,0,0.30,"Austin, Texas"


In [362]:
data_airbnb['price_clean'] = (
    data_airbnb['price']
    .astype(str)
    .str.replace(r'[$,]', '', regex=True)
    .str.strip()
)

data_airbnb['price_clean'] = pd.to_numeric(data_airbnb['price_clean'], errors='coerce')

data_airbnb = data_airbnb[
    data_airbnb['price_clean'].notna() &
    (data_airbnb['price_clean'] > 0)
].copy()

In [363]:
q1 = data_airbnb['price_clean'].quantile(0.25)
q3 = data_airbnb['price_clean'].quantile(0.75)
iqr = q3 - q1

data_airbnb = data_airbnb[
    data_airbnb['price_clean'] <= (q3 + 1.5 * iqr)
].copy()

print("Shape después de limpiar:", data_airbnb.shape)
print(data_airbnb['price_clean'].describe())

Shape después de limpiar: (68737, 81)
count    68737.000000
mean       209.363458
std        128.664537
min          8.000000
25%        114.000000
50%        177.000000
75%        274.000000
max        635.000000
Name: price_clean, dtype: float64


In [364]:
features = [
    'accommodates',
    'bedrooms',
    'beds',
    'review_scores_rating',
    'number_of_reviews'
]

features = [f for f in features if f in data_airbnb.columns]

X = data_airbnb[features].copy()
y_real = data_airbnb['price_clean'].copy()

print("Features usadas:", features)
print("Shape X:", X.shape)

Features usadas: ['accommodates', 'bedrooms', 'beds', 'review_scores_rating', 'number_of_reviews']
Shape X: (68737, 5)


In [365]:
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=features,
    index=X.index
)

C:\Users\belen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\Users\belen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\_core\fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [366]:
q_bajo = y_real.quantile(0.33)
q_alto = y_real.quantile(0.66)

bins = [0, q_bajo, q_alto, y_real.max()]
labels = [0, 1, 2]

y_class = pd.cut(
    y_real,
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

print(y_class.value_counts().sort_index())

price_clean
0    22870
1    22532
2    23335
Name: count, dtype: int64


In [367]:
X_train, X_test, y_train_real, y_test_real, y_train_class, y_test_class = train_test_split(
    X,
    y_real,
    y_class,
    test_size=0.30,
    random_state=42
)

In [368]:
scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=features,
    index=X_train.index
)

X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=features,
    index=X_test.index
)

# 1. Modelo de regresión usando bayes

In [369]:
modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train_class)

y_pred_class = modelo_nb.predict(X_test)

accuracy = accuracy_score(y_test_class, y_pred_class)
print(f'Precisión del modelo: {accuracy:.2f}')

Precisión del modelo: 0.52
